In [50]:
#Set up paths
import os
import sys

# 1. Get the current working directory of the Jupyter Notebook
current_dir = os.getcwd()

# 2. Get the path to 'Kitaev-Diagnolization' (parent of the current dir)
project_root = os.path.dirname(current_dir)

# 3. Add it to the system path
sys.path.append(project_root)


In [51]:
import importlib
import math
import numpy as np
import sympy as sp
from IPython.display import display
from tqdm import tqdm

import static.unit_cell as consts
import src.placket.generate_relation_matrix as grm 
import src.core.base_matrix as base_mtrx

importlib.reload(grm)
importlib.reload(consts)
importlib.reload(base_mtrx)

#Pull the updated specific classes into your namespace
from src.placket.generate_relation_matrix import Relation_Table
from src.core.base_matrix import BaseMatrix



In [52]:
BaseMatrix.update_lib()

The letter of the update is f


### Important Code Conventions & Mapping

Before diving into the logic, it is crucial to understand the exact hierarchy of concepts and how the physical model translates to the programmatic variables.

---

#### 1. The Primacy of Sites and Alpha Bonds
* **Global Sites are Fundamental:** The absolute most important variables are the global site numbers. Every other identifier (unit cells, sublattices, hopping directions) is strictly derived from these base connections.
* **Alpha Bonds ($\alpha$):** The second most critical components are the geometric alpha bonds, which define exactly how these sites are physically connected. In the code, these don't appear as matrix indices; instead, they are treated as geometric constants that are looped over inside `f_vector` to construct the actual mathematical terms for the summation.

#### 2. The $f$ Notation: A Useful Shorthand
* **Unit Cell Jumping:** The $f_{xy}$ notation is essentially a shorthand for unit cell jumping (e.g., $f_{12}$ represents the connection terms between unit cell `1` and unit cell `2`). 
* **Array Offset:** Because we use 1-based indexing for the physics (to match the literature) but Python uses native 0-based arrays, you must apply an offset to this shorthand. Accessing the conceptual $f_{12}$ connection requires pulling from the array at `f_vector[0][1]`.

#### 3. Directionality and the Hermitian Requirement
* **Resolving Ambiguity:** The unit cell shorthand ($f_{12}$) creates an inherent ambiguity: is the particle hopping from $1 \to 2$ or $2 \to 1$? 
* **The Complex Conjugate ($^*$):** This ambiguity is resolved entirely by the directionality argument. The physical hopping direction does **not** change the underlying summation terms (the $\alpha$ bonds inside $f_{12}$ are identical to those inside $f_{12}^*$). Instead, directionality dictates whether we apply a complex conjugate.
* **Hermitian Preservation:** If the hop is moving "backward" (a $B \to A$ transition), the code applies a negative complex conjugate ($-f_{xy}^*$). This operation is absolutely pivotal to ensure the resulting Hamiltonian matrix remains strictly Hermitian.

In [53]:
#Here are the inputs of the function
SitesPerCell = 2
TotalNumberOfSites = 4
f_vectors = consts.hexagon_sites.FourSite
translation_vectors = consts.hexagon_translation.FourSiteLinear
D = 2 #number of dimensions
stacked_by = "row" #tells us wether we are stacking our f_vector matrix and translation_vectors row-wise or column wise. This is important for 
u_ij = {"site": {2,3}, "bond_perp_basis": (-np.sqrt(3)/2,-1/2)} #in the form of which_sites_are_being_connected:{site1,site2}, how_those_sites_are_connected(x, y,...), and keep in mind we dont care about the f_index so it is a SET, but we do care about order for the vector(x,y)

In [54]:
# CHECK CELL YOU CAN IGNORE THIS
print("=" * 45)
print("Coupling Cell Vectors Configuration")
print("=" * 45)

# Cleaned up Full Set print
print("Full Set of Vectors:")
for row in f_vectors:
    # str() removes the 'array()' wrapper, and replace() keeps blocks on one line
    row_strings = [str(block).replace('\n', ' ') for block in row]
    print(f"[ { '   |   '.join(row_strings) } ] ")
print("-" * 45)

# Individual blocks
print(f"f_11 connections:\n{f_vectors[0,0]}\n")
print(f"f_12 connections:\n{f_vectors[0,1]}\n")
print(f"f_21 connections:\n{f_vectors[1,0]}\n")
print(f"f_22 connections:\n{f_vectors[1,1]}")
print("=" * 45)

Coupling Cell Vectors Configuration
Full Set of Vectors:
[ [[0 0]]   |   [[-0.8660254  0.5      ]  [-0.8660254 -0.5      ]] ] 
[ [[-0.8660254  0.5      ]  [-0.8660254 -0.5      ]]   |   [[0 0]] ] 
---------------------------------------------
f_11 connections:
[[0 0]]

f_12 connections:
[[-0.8660254  0.5      ]
 [-0.8660254 -0.5      ]]

f_21 connections:
[[-0.8660254  0.5      ]
 [-0.8660254 -0.5      ]]

f_22 connections:
[[0 0]]


### The Code Flow: Step-by-Step

Here is exactly how the main loop builds the Hamiltonian matrices from the ground up:

#### 1. Looping & Vector Selection
* **The Grid:** We loop over every possible combination of `site_start` and `site_end`.
* **Filtering:** If a transition stays on the exact same sublattice (e.g., $A_1 \to A_1$), we skip it, as these diagonal elements are strictly zero.
* **Fetching Vectors:** Using the unit cell and direction, we grab the raw geometric hopping bonds (`f_vector`) that connect the two sites in real space.

#### 2. Basis Transformation (Aligning the Brillouin Zone)
* **The Problem:** Raw hopping vectors are in standard Cartesian ($x,y$) space, which makes momentum math messy.
* **The Fix:** `BaseMatrix.convert_basis(f_vector,translation_vectors,stacked_by=stacked_by,symbolic=True)` - Converts these vectors into the **translation basis** of the lattice. 
* **The Benefit:** This perfectly aligns the momentum variables ($k_1, k_2$) with the boundaries of the Brillouin zone, allowing them to scale cleanly from $0 \to 1$ without needing irrational trigonometric constants.

#### 3. Generating the Phase (The Dot Product)
* **The $U_{ij}$ Check:** We loop over each $\alpha$ bond, checking if it matches our specific interaction site. If it does, we apply the interaction flip (`u_value = -1`).
* **The Math:** We generate the phase shift by taking the dot product of the momentum $k$ and the transformed bond vector, yielding the term $e^{i (k \cdot \alpha)}$.
* **Directionality:** If the hop is a reverse transition ($B \to A$), we wrap the result in a negative complex conjugate to ensure the final matrix is Hermitian.

#### 4. Populating the Matrices
* **$H_{1/2}$ (Half Matrix):** The computed positive momentum ($k$) values are slotted directly into their corresponding row and column.
* **$H_{\text{full}}$ (Folded Matrix):** The full $2N \times 2N$ matrix is built dynamically. The $k$ sector is placed in the top-left, and the corresponding $-k$ sector is calculated and placed in the bottom-right.

In [55]:
# --- DEBUG TOGGLES & PRINTERS ---
DEBUG_LINKS = True  # Set to False to hide general link prints
DEBUG_UIJ = True    # Set to False to hide U_ij match prints

def debug_link(start, end, flip, u_start, u_end, sub_start, sub_end, f_vec, f_vec_t):
    if DEBUG_LINKS:
        print(f"=== Link: Global Site {start} -> {end} ===")
        print(f"    Flip Dir  : {flip}")
        print(f"    Unit Cell : {u_start} -> {u_end}")
        print(f"    Sublattice: {sub_start} -> {sub_end}")
        print(f"    f_vector  : {f_vec}")
        print(f"    f_vectorT : {f_vec_t}\n")

def debug_uij(start, end, vector, flip, f_val):
    if DEBUG_UIJ:
        print(f"{'='*60}\nU_ij match found!")
        print(f"Site      : {start} -> {end}")
        print(f"Vector    : {np.round(np.array(vector, dtype=float), 3).tolist()}")
        print(f"Flip Dir  : {flip}\nf_{start},{end} :")
        display(f_val)
        print(f"{'='*60}\n")
# --------------------------------

In [56]:
#create the Hamiltonian matrix
H_half_check = np.zeros([TotalNumberOfSites,TotalNumberOfSites])
H_full_check =  np.zeros([2*TotalNumberOfSites,2*TotalNumberOfSites])
H_half = np.zeros([TotalNumberOfSites,TotalNumberOfSites], dtype=object)
H_full = np.zeros([2*TotalNumberOfSites,2*TotalNumberOfSites], dtype=object) 

#MAIN LOGIC
#this goes over all the site combintations
for site_start in tqdm(range(1, TotalNumberOfSites+1), desc="Building Hamiltonian"):
    for site_end in range(1, TotalNumberOfSites+1):
        
        #figure out which site you are at, and where in the site you are at for both your starting and ending value
        unit_cell_start = (math.ceil(site_start / SitesPerCell)) 
        unit_cell_end = (math.ceil(site_end / SitesPerCell))
        inside_unit_cell_start = site_start % SitesPerCell  
        inside_unit_cell_end = site_end % SitesPerCell

        #if you are going to the same inside site (A1 -> A2) then skip the value, since it will be a 0 value in the Hamiltonian (alredy set!)
        if inside_unit_cell_start == inside_unit_cell_end:
            continue

        #if it takes us to a diffrent inter-cell-site then we have to do some computations

        #if flip is TRUE  we are starting on a B(1) site
        #if flip is FALSE we are starting on a A(0) site
        flip = True if inside_unit_cell_start < inside_unit_cell_end else False

        #####SPECIFIC TO TWO SITE CELLS####
        #find the right corresponding f vectors
        f_vector = f_vectors[unit_cell_end-1][unit_cell_start-1] if flip else f_vectors[unit_cell_start-1][unit_cell_end-1]
        f_vector_translation_basis_sympy = BaseMatrix.convert_basis(f_vector,translation_vectors,stacked_by=stacked_by,symbolic=True) #make the vector in the translation basis as well
        f_vector_translation_basis_numpy = np.array (f_vector_translation_basis_sympy)
        # Call the helper function instead of cluttering the loop
        debug_link(site_start, site_end, flip, unit_cell_start, unit_cell_end, 
                   inside_unit_cell_start, inside_unit_cell_end, f_vector, f_vector_translation_basis_numpy)
        #####SPECIFIC TO TWO SITE CELLS####


        #Logic for building H
        f_vector = f_vector if stacked_by == "row" else f_vector.T
        for idx, vector_perp_basis in enumerate(f_vector):
            vector_translation_basis = f_vector_translation_basis_numpy[idx] if stacked_by == "row" else f_vector_translation_basis_numpy[:, idx]            # vector_translation_basis = vector_perp_basis #testing

            u_value = -1 if u_ij["site"] == {site_start,site_end} and np.allclose(vector_perp_basis, u_ij["bond_perp_basis"]) else 1
            
            #compute the normal values first
            if flip: #if "flipped" means its a B to A site which needs us to do f^*
                f       =  sp.conjugate( sp.I * u_value *  sp.E ** BaseMatrix.dot_prod(2,"k",vector_translation_basis))
                f_k_neg =  sp.conjugate( sp.I * u_value *  sp.E ** BaseMatrix.dot_prod(2,"-k",vector_translation_basis))

            else:
                f       = sp.I * (u_value) * sp.E ** BaseMatrix.dot_prod(2,"k", vector_translation_basis)
                f_k_neg = sp.I * (u_value) * sp.E ** BaseMatrix.dot_prod(2,"-k", vector_translation_basis)

            if u_value == -1:
                debug_uij(site_start, site_end, vector_perp_basis, flip, f)


            #Build the half check matrix to check if you are updating the right spots
            H_half_check[site_start -1, site_end-1] += 1
            #Build the full check matrix
            H_full_check[site_start -1, site_end-1] += 1 
            H_full_check[2*TotalNumberOfSites - site_start, 2*TotalNumberOfSites - site_end] += -1

            #Build the half matrix to make sure that we got half of it right
            H_half[site_start -1 , site_end-1] += f            

            #Build the full matrix by folding out the values 
            H_full[site_start -1 , site_end -1 ] += f
            H_full[2*TotalNumberOfSites - site_start, 2*TotalNumberOfSites - site_end] += f_k_neg

# #display whatever you want to see
# display(sp.Matrix(H_half_check))
# display(sp.Matrix(H_full_check))

# display(sp.Matrix(H_half))
# display(sp.Matrix(H_full))



Building Hamiltonian:   0%|          | 0/4 [00:00<?, ?it/s]

=== Link: Global Site 1 -> 2 ===
    Flip Dir  : False
    Unit Cell : 1 -> 1
    Sublattice: 1 -> 0
    f_vector  : [[0 0]]
    f_vectorT : [[0 0]]

=== Link: Global Site 1 -> 4 ===
    Flip Dir  : False
    Unit Cell : 1 -> 2
    Sublattice: 1 -> 0
    f_vector  : [[-0.8660254  0.5      ]
 [-0.8660254 -0.5      ]]
    f_vectorT : [[1.73205080756888 0.500000000000000]
 [1.11022302462516e-16 -0.500000000000000]]

=== Link: Global Site 2 -> 1 ===
    Flip Dir  : True
    Unit Cell : 1 -> 1
    Sublattice: 0 -> 1
    f_vector  : [[0 0]]
    f_vectorT : [[0 0]]

=== Link: Global Site 2 -> 3 ===
    Flip Dir  : True
    Unit Cell : 1 -> 2
    Sublattice: 0 -> 1
    f_vector  : [[-0.8660254  0.5      ]
 [-0.8660254 -0.5      ]]
    f_vectorT : [[1.73205080756888 0.500000000000000]
 [1.11022302462516e-16 -0.500000000000000]]

U_ij match found!
Site      : 2 -> 3
Vector    : [-0.866, -0.5]
Flip Dir  : True
f_2,3 :


I*exp(-2*I*pi*(27755575615629*conjugate(k1)/250000000000000000000000000000 - conjugate(k2)/2))


=== Link: Global Site 3 -> 2 ===
    Flip Dir  : False
    Unit Cell : 2 -> 1
    Sublattice: 1 -> 0
    f_vector  : [[-0.8660254  0.5      ]
 [-0.8660254 -0.5      ]]
    f_vectorT : [[1.73205080756888 0.500000000000000]
 [1.11022302462516e-16 -0.500000000000000]]

U_ij match found!
Site      : 3 -> 2
Vector    : [-0.866, -0.5]
Flip Dir  : False
f_3,2 :


-I*exp(2*I*pi*(27755575615629*k1/250000000000000000000000000000 - k2/2))

Building Hamiltonian: 100%|██████████| 4/4 [00:00<00:00, 56.56it/s]


=== Link: Global Site 3 -> 4 ===
    Flip Dir  : False
    Unit Cell : 2 -> 2
    Sublattice: 1 -> 0
    f_vector  : [[0 0]]
    f_vectorT : [[0 0]]

=== Link: Global Site 4 -> 1 ===
    Flip Dir  : True
    Unit Cell : 2 -> 1
    Sublattice: 0 -> 1
    f_vector  : [[-0.8660254  0.5      ]
 [-0.8660254 -0.5      ]]
    f_vectorT : [[1.73205080756888 0.500000000000000]
 [1.11022302462516e-16 -0.500000000000000]]

=== Link: Global Site 4 -> 3 ===
    Flip Dir  : True
    Unit Cell : 2 -> 2
    Sublattice: 0 -> 1
    f_vector  : [[0 0]]
    f_vectorT : [[0 0]]



In [63]:
TotalKSites = 100
k1_vals, k2_vals = np.meshgrid(np.arange(0,1,1/np.sqrt(TotalKSites)), np.arange(0,1,1/np.sqrt(TotalKSites)))
H_full_real = sp.Matrix(H_full).subs({sp.Symbol('k1'): sp.Symbol('k1', real=True),sp.Symbol('k2'): sp.Symbol('k2', real=True)})
H_full_numerical = BaseMatrix.symbolic_to_numerical_matrix(sp.Matrix(H_full_real), {'k1':k1_vals, 'k2':k2_vals})

In [64]:
# Energy Calculations
E0 = -1.5746
TotalEnergy = BaseMatrix.sum_neg_eignvalues(H_full_numerical)
TotalEnergyPerSite = TotalEnergy / (TotalKSites * TotalNumberOfSites)
GroundStateEnergy = -TotalEnergy / 2  # Assuming particle-hole symmetry correction

In [65]:
# --- FORMATTED OUTPUT ---
print(f"\n{'='*60}")
print(" ENERGY CALCULATION SUMMARY ".center(60, '='))
print(f"{'='*60}")
print(f"Total k-points sampled       : {TotalKSites}")
print(f"Total Physical Sites         : {TotalNumberOfSites}")
print(f"Reference Base Energy (E0)   : {E0}")
print(f"{'-'*60}")
print(f"Ground State Energy (-E/2)   : {GroundStateEnergy:.6f}")
print(f"Total Energy Per Site        : {TotalEnergyPerSite:.6f}")
print(f"Relative Energy (vs E0)      : {(TotalEnergyPerSite - E0):.6f}")
print(f"{'='*60}\n")


================ ENERGY CALCULATION SUMMARY ================
Total k-points sampled       : 100
Total Physical Sites         : 4
Reference Base Energy (E0)   : -1.5746
------------------------------------------------------------
Ground State Energy (-E/2)   : 302.039103
Total Energy Per Site        : -1.510196
Relative Energy (vs E0)      : 0.064404

